# Notebook 1: Text extraction and chunking

**Goal:** see exactly what happens to a document between upload and the stored chunks, before we build the API. The notebook uses the same code the service will use (`app/ingestion`).

**Before you start**
1. Put at least one PDF and one Word file in `data/sample_docs/`. Any files will do for now.
2. Select the `.venv` kernel (top right in VS Code), then run the cells from the top.

In [8]:
import re
import statistics
import sys
from pathlib import Path

# Make the project importable when the notebook runs from the notebooks/ folder.
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from app.ingestion.chunk import chunk_pages
from app.ingestion.parse import parse_file

DOCS = ROOT / "data" / "sample_docs"
pdfs = sorted(DOCS.glob("*.pdf"))
docxs = sorted(DOCS.glob("*.docx"))
print(f"Found {len(pdfs)} PDF and {len(docxs)} Word files in {DOCS}")

Found 1 PDF and 1 Word files in C:\Users\Dell\Downloads\rag-chatbot-step1\rag-chatbot\data\sample_docs


## 1. Extract a PDF
Each page becomes Markdown (headings, lists, tables) together with its page number. Page numbers are what the citations will show.

In [9]:
pdf_pages = parse_file(pdfs[0]) if pdfs else []
if pdf_pages:
    print(f"{pdfs[0].name}: {len(pdf_pages)} pages")
    for page in pdf_pages[:10]:
        print(f"  page {page.page}: {len(page.text):>6} characters")
    first = next((p for p in pdf_pages if p.text.strip()), None)
    print("\n--- preview of the first page with text ---\n")
    print(first.text[:1200] if first else "(no text found)")
else:
    print("Add a PDF to data/sample_docs/ and run this cell again.")


=== Document parser messages ===
Deputy_Demographer_BPS17_Complete_Preparation_Guide.pdf: 77 pages
  page 1:   1136 characters
  page 2:   3689 characters
  page 3:   3867 characters
  page 4:    749 characters
  page 5:   3074 characters
  page 6:   1378 characters
  page 7:   3161 characters
  page 8:   3397 characters
  page 9:    826 characters
  page 10:   2873 characters

--- preview of the first page with text ---

# **COMPLETE COMPETITIVE EXAMINATION PREPARATION GUIDE**

DEPUTY DEMOGRAPHER (BPS-17) Khyber Pakhtunkhwa Public Service Commission

**Population Welfare Department | Advertisement No. 02/2026 | Serial No. 45**

**Purpose:** A structured beginner-to-advanced preparation book covering demography, population welfare, statistics, census, research methods, economics, sociology, gender, Pakistan/KP facts, numerical practice and five 45-question mock examinations.

Prepared from the uploaded 13-page syllabus and supplemented with official-source verification where available

### Look for problems
Pages with almost no text are usually scans (images of text). The service cannot read those yet, so we need to know how common they are in the corpus.

In [10]:
thin_pages = [p.page for p in pdf_pages if len(p.text.strip()) < 50]
print("Pages with almost no text:", thin_pages or "none")
table_count = sum(len(re.findall(r"^\|\s*:?-{3,}", p.text, flags=re.M)) for p in pdf_pages)
print("Markdown tables detected:", table_count)

Pages with almost no text: none
Markdown tables detected: 49


## 2. Extract a Word file
Word files have no fixed pages, so the page number is `None` and the headings identify the location instead.

In [11]:
docx_pages = parse_file(docxs[0]) if docxs else []
if docx_pages:
    text = docx_pages[0].text
    print(f"{docxs[0].name}: {len(text)} characters, page number: {docx_pages[0].page}")
    print("Tables:", sum(1 for line in text.split("\n") if line.startswith("| ---")))
    print("\n--- preview ---\n")
    print(text[:1200])
else:
    print("Add a .docx file to data/sample_docs/ and run this cell again.")

Portfolio_Project_1_Practical_RAG_Chatbot_Guide.docx: 15703 characters, page number: None
Tables: 2

--- preview ---

# Portfolio Project 1

Practical Step-by-Step Build Guide: Document-Based RAG Chatbot

Goal: Build a chatbot that lets users upload their documents, ask questions, and receive evidence-based answers with source citations.

# Project outcome

By following this guide, I will build a working RAG chatbot incrementally. I will first create a simple baseline, test it, and then add retrieval improvements, citations, evaluation, and security. I will avoid implementing every advanced feature at once.

# Final technology stack

| Component | Initial choice | Why |
| --- | --- | --- |
| UI | Streamlit | Fast Python-based upload and chat interface |
| Backend | Python modules (FastAPI optional initially) | Keep the first version easy to run; add API separation later |
| LLM | GPT-4.1 mini (or currently available equivalent) | Good quality/cost balance for grounded Q&A |
| Embedding

## 3. Chunk
The chunker splits on headings, then paragraphs, then sentences. Every chunk keeps its heading path and page range. The defaults are in `app/config.py` (about 500 tokens, 60 tokens of overlap).

In [12]:
pages = pdf_pages or docx_pages
chunks = chunk_pages(pages) if pages else []
if chunks:
    sizes = [c.token_count for c in chunks]
    print(f"{len(chunks)} chunks | tokens: min {min(sizes)}, median {int(statistics.median(sizes))}, max {max(sizes)}")
    for chunk in chunks[:3]:
        print("\n" + "=" * 70)
        print(f"chunk {chunk.index} | pages {chunk.page_start}-{chunk.page_end} | {chunk.token_count} tokens")
        print(f"heading: {chunk.heading}")
        print(chunk.text[:500] + (" ..." if len(chunk.text) > 500 else ""))
else:
    print("Nothing to chunk yet.")

118 chunks | tokens: min 121, median 256, max 761

chunk 0 | pages 1-1 | 123 tokens
heading: COMPLETE COMPETITIVE EXAMINATION PREPARATION GUIDE
DEPUTY DEMOGRAPHER (BPS-17) Khyber Pakhtunkhwa Public Service Commission

**Population Welfare Department | Advertisement No. 02/2026 | Serial No. 45**

**Purpose:** A structured beginner-to-advanced preparation book covering demography, population welfare, statistics, census, research methods, economics, sociology, gender, Pakistan/KP facts, numerical practice and five 45-question mock examinations.

Prepared from the uploaded 13-page syllabus and supplemented with official-source verification  ...

chunk 1 | pages 1-2 | 139 tokens
heading: COMPLETE COMPETITIVE EXAMINATION PREPARATION GUIDE > EXAM FORMAT IN UPLOADED SYLLABUS
|**Item**|**Detail**|
|---|---|
|Questions|45 multiple-choice questions|
|Duration|45 minutes|
|Test date shown in uploaded syllabus|Thursday, 26 November 2026|
|Reporting / test time shown|2:00 PM reporting; 3:00–3:45 PM 

### Check by eye
Read the chunks above. Does each one make sense on its own? Is the heading right? Is a table cut in half? Write what you see in the notes at the bottom, because it goes into DESIGN.md.

In [13]:
if chunks:
    smallest = min(chunks, key=lambda c: c.token_count)
    largest = max(chunks, key=lambda c: c.token_count)
    for label, chunk in (("Smallest", smallest), ("Largest", largest)):
        print(f"\n{label} chunk ({chunk.token_count} tokens, heading: {chunk.heading})")
        print(chunk.text[:400])


Smallest chunk (121 tokens, heading: COMPLETE COMPETITIVE EXAMINATION PREPARATION GUIDE > Chapter 7. Statistics and Demographic Techniques > 7.4 Worked statistics examples)
Data: 2, 4, 4, 6, 9. Mean = 25/5 = 5. Median = 4. Mode = 4. Range = 9−2 = 7. Squared deviations from mean: 9, 1, 1, 1, 16; sum = 28. Population variance = 28/5 = 5.6 and population SD = √5.6 ≈ 2.37. Sample variance = 28/4 = 7 and sample SD ≈ 2.65.

Largest chunk (761 tokens, heading: COMPLETE COMPETITIVE EXAMINATION PREPARATION GUIDE > Chapter 11. Formula Handbook and Numerical Practice)
|**Measure**<br>Crude Birth Rate|**Formula**<br>CBR = (live births / mid-year population) × 1,000|**Interpretation / units**<br>Births per 1,000 total population per year.|
|---|---|---|
|Crude Death Rate|CDR = (deaths / mid-year population) × 1,000|Deaths per 1,000 total population per year.|
|General Fertility Rate|GFR = (live births / women aged 15–49) × 1,000|Births per 1,000 women aged 15–49.


## 4. First experiment: chunk size
How many chunks do different sizes produce? The retrieval metrics (later) decide which size is actually best; for now we only see the effect on the number of chunks.

In [14]:
if pages:
    for target in (256, 512, 1024):
        trial = chunk_pages(
            pages,
            target_tokens=target,
            max_tokens=int(target * 1.6),
            overlap_tokens=int(target * 0.12),
            min_tokens=int(target * 0.25),
        )
        sizes = [c.token_count for c in trial]
        print(f"target {target:>4}: {len(trial):>4} chunks, mean {statistics.mean(sizes):.0f} tokens")

target  256:  196 chunks, mean 204 tokens
target  512:  115 chunks, mean 342 tokens
target 1024:   66 chunks, mean 589 tokens


## Notes for DESIGN.md (fill these in)
- Which file types extracted cleanly?
- Any scanned pages or broken tables?
- Chunks that looked wrong, and why?
- Which chunk size looks best so far? (Retrieval metrics decide later.)